In [1]:
from pathlib import Path

import kagglehub
import pandas as pd

dataset_dir = kagglehub.dataset_download(
    "shivamb/real-or-fake-fake-jobposting-prediction"
)

csv_path = Path(dataset_dir) / "fake_job_postings.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
df[["title", "description", "fraudulent"]].head()

Using Colab cache for faster access to the 'real-or-fake-fake-jobposting-prediction' dataset.
Dataset shape: (17880, 18)


,title,description,fraudulent
0,Marketing Intern,"Food52, a fast-growing, James Beard Award-winn...",0
1,Customer Service - Cloud Video Production,Organised - Focused - Vibrant - Awesome!Do you...,0
2,Commissioning Machinery Assistant (CMA),"Our client, located in Houston, is actively se...",0
3,Account Executive - Washington DC,THE COMPANY: ESRI – Environmental Systems Rese...,0
4,Bill Review Manager,JOB TITLE: Itemization Review ManagerLOCATION:...,0


In [2]:
class_counts = df["fraudulent"].value_counts().sort_index()

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": (class_counts / class_counts.sum() * 100).round(2)
})

class_summary = class_summary.rename(index={0: "Real", 1: "Fake"})
class_summary

,Count,Percentage
fraudulent,,
Real,17014,95.16
Fake,866,4.84


In [3]:
missing_counts = df.isna().sum().sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "Missing count": missing_counts,
    "Missing percentage": (missing_counts / len(df) * 100).round(2)
})

display(missing_summary.head(8))

duplicate_count = df.drop(columns="job_id").duplicated().sum()
print("Duplicate rows excluding job_id:", duplicate_count)

,Missing count,Missing percentage
salary_range,15012,83.96
department,11547,64.58
required_education,8105,45.33
benefits,7212,40.34
required_experience,7050,39.43
function,6455,36.10
industry,4903,27.42
employment_type,3471,19.41


Duplicate rows excluding job_id: 281


In [4]:
comparison_columns = df.columns.drop("job_id")
clean_df = df.drop_duplicates(subset=comparison_columns).copy()

print("Rows before cleaning:", len(df))
print("Rows after removing duplicates:", len(clean_df))
print("Missing descriptions:", clean_df["description"].isna().sum())


Rows before cleaning: 17880
Rows after removing duplicates: 17599
Missing descriptions: 1


In [5]:
model_df = clean_df.dropna(subset=["description"])

X = model_df["description"]
y = model_df["fraudulent"]

print("Rows available for modeling:", len(model_df))
print("Input samples:", len(X))
print("Target labels:", len(y))

Rows available for modeling: 17598
Input samples: 17598
Target labels: 17598


In [6]:
import numpy as np
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors

normalized_descriptions = (
    model_df["description"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)

grouping_matrix = TfidfVectorizer().fit_transform(normalized_descriptions)
similarity_graph = NearestNeighbors(
    radius=0.10, metric="cosine", algorithm="brute"
).fit(grouping_matrix).radius_neighbors_graph(grouping_matrix, mode="connectivity")

group_count, group_labels = connected_components(similarity_graph, directed=False)
description_groups = pd.Series(group_labels, index=model_df.index)

group_sizes = description_groups.value_counts()
print("Number of similarity groups:", group_count)
print("Largest group size:", group_sizes.iloc[0])
print("Groups with more than one row:", int((group_sizes > 1).sum()))

group_splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42)
fold_ids = pd.Series(-1, index=model_df.index)
for fold_number, (_, fold_positions) in enumerate(
    group_splitter.split(model_df, model_df["fraudulent"], groups=description_groups)
):
    fold_ids.iloc[fold_positions] = fold_number

train_index = fold_ids.index[fold_ids < 14]
val_index = fold_ids.index[(fold_ids >= 14) & (fold_ids < 17)]
test_index = fold_ids.index[fold_ids >= 17]

X_train, y_train = X.loc[train_index], y.loc[train_index]
X_val, y_val = X.loc[val_index], y.loc[val_index]
X_test, y_test = X.loc[test_index], y.loc[test_index]

split_groups = {
    "Training": set(description_groups.loc[train_index]),
    "Validation": set(description_groups.loc[val_index]),
    "Test": set(description_groups.loc[test_index]),
}

for first, second in [("Training", "Validation"), ("Training", "Test"), ("Validation", "Test")]:
    shared = len(split_groups[first] & split_groups[second])
    print(f"Shared groups {first} - {second}: {shared}")
    assert shared == 0

total_rows = len(X_train) + len(X_val) + len(X_test)
print("Total rows:", total_rows)
assert total_rows == len(model_df) == 17598
assert (fold_ids >= 0).all()

for name, labels in [("Training", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(
        f"{name}: {len(labels)} samples "
        f"({len(labels) / total_rows * 100:.2f}%), {labels.mean() * 100:.2f}% fake"
    )

Number of similarity groups: 12372
Largest group size: 414
Groups with more than one row: 1767
Shared groups Training - Validation: 0
Shared groups Training - Test: 0
Shared groups Validation - Test: 0
Total rows: 17598
Training: 11850 samples (67.34%), 5.03% fake
Validation: 3151 samples (17.91%), 4.06% fake
Test: 2597 samples (14.76%), 5.04% fake


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline

baseline_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000)
)

baseline_model.fit(X_train, y_train)
validation_predictions = baseline_model.predict(X_val)

print(classification_report(
    y_val,
    validation_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, validation_predictions, labels=[0, 1]))

              precision    recall  f1-score   support

        Real      0.961     1.000     0.980      3023
        Fake      1.000     0.039     0.075       128

    accuracy                          0.961      3151
   macro avg      0.980     0.520     0.528      3151
weighted avg      0.962     0.961     0.943      3151

Confusion matrix:
[[3023    0]
 [ 123    5]]


In [8]:
weighted_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

weighted_model.fit(X_train, y_train)
weighted_predictions = weighted_model.predict(X_val)

print(classification_report(
    y_val,
    weighted_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, weighted_predictions, labels=[0, 1]))

              precision    recall  f1-score   support

        Real      0.982     0.969     0.976      3023
        Fake      0.446     0.586     0.507       128

    accuracy                          0.954      3151
   macro avg      0.714     0.778     0.741      3151
weighted avg      0.960     0.954     0.957      3151

Confusion matrix:
[[2930   93]
 [  53   75]]


In [9]:
from sklearn.naive_bayes import MultinomialNB

naive_bayes_model = make_pipeline(
    TfidfVectorizer(),
    MultinomialNB()
)

naive_bayes_model.fit(X_train, y_train)
naive_bayes_predictions = naive_bayes_model.predict(X_val)

print(classification_report(
    y_val,
    naive_bayes_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, naive_bayes_predictions, labels=[0, 1]))

              precision    recall  f1-score   support

        Real      0.959     0.999     0.979      3023
        Fake      0.000     0.000     0.000       128

    accuracy                          0.958      3151
   macro avg      0.480     0.500     0.489      3151
weighted avg      0.920     0.958     0.939      3151

Confusion matrix:
[[3020    3]
 [ 128    0]]


In [10]:
from sklearn.svm import LinearSVC

svm_model = make_pipeline(
    TfidfVectorizer(),
    LinearSVC(random_state=42)
)

weighted_svm_model = make_pipeline(
    TfidfVectorizer(),
    LinearSVC(class_weight="balanced", random_state=42)
)

for name, model in [
    ("Linear SVM", svm_model),
    ("Weighted Linear SVM", weighted_svm_model)
]:
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    print(f"\n{name}")
    print(classification_report(
        y_val,
        predictions,
        labels=[0, 1],
        target_names=["Real", "Fake"],
        digits=3,
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_val, predictions, labels=[0, 1]))


Linear SVM
              precision    recall  f1-score   support

        Real      0.965     1.000     0.982      3023
        Fake      0.952     0.156     0.268       128

    accuracy                          0.965      3151
   macro avg      0.959     0.578     0.625      3151
weighted avg      0.965     0.965     0.953      3151

Confusion matrix:
[[3022    1]
 [ 108   20]]

Weighted Linear SVM
              precision    recall  f1-score   support

        Real      0.980     0.991     0.985      3023
        Fake      0.705     0.523     0.601       128

    accuracy                          0.972      3151
   macro avg      0.843     0.757     0.793      3151
weighted avg      0.969     0.972     0.970      3151

Confusion matrix:
[[2995   28]
 [  61   67]]


In [11]:
from sklearn.metrics import f1_score

validation_scores = weighted_svm_model.decision_function(X_val)

for threshold in [-0.50, -0.25, -0.10, 0.00, 0.10, 0.25, 0.50]:
    predictions = (validation_scores > threshold).astype(int)

    _, false_positives, false_negatives, true_positives = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    fake_f1 = f1_score(y_val, predictions, pos_label=1)

    print(
        f"Threshold {threshold:+.2f} | Fake F1 {fake_f1:.3f} | "
        f"Detected {true_positives}/{len(y_val[y_val == 1])} | "
        f"Missed {false_negatives} | False alarms {false_positives}"
    )

Threshold -0.50 | Fake F1 0.461 | Detected 97/128 | Missed 31 | False alarms 196
Threshold -0.25 | Fake F1 0.551 | Detected 76/128 | Missed 52 | False alarms 72
Threshold -0.10 | Fake F1 0.606 | Detected 73/128 | Missed 55 | False alarms 40
Threshold +0.00 | Fake F1 0.601 | Detected 67/128 | Missed 61 | False alarms 28
Threshold +0.10 | Fake F1 0.548 | Detected 57/128 | Missed 71 | False alarms 23
Threshold +0.25 | Fake F1 0.333 | Detected 27/128 | Missed 101 | False alarms 7
Threshold +0.50 | Fake F1 0.245 | Detected 18/128 | Missed 110 | False alarms 1


## Experiment 2: Title, Company Profile, and Description

In [12]:
text_columns = ["title", "company_profile", "description"]
combined_text = model_df[text_columns].fillna("").agg(" ".join, axis=1)

X_train_combined = combined_text.loc[X_train.index]
X_val_combined = combined_text.loc[X_val.index]

combined_model = make_pipeline(
    TfidfVectorizer(),
    LinearSVC(class_weight="balanced", random_state=42)
)

combined_model.fit(X_train_combined, y_train)
combined_predictions = combined_model.predict(X_val_combined)

print(classification_report(
    y_val,
    combined_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, combined_predictions, labels=[0, 1]))

              precision    recall  f1-score   support

        Real      0.987     0.998     0.992      3023
        Fake      0.946     0.680     0.791       128

    accuracy                          0.985      3151
   macro avg      0.966     0.839     0.892      3151
weighted avg      0.985     0.985     0.984      3151

Confusion matrix:
[[3018    5]
 [  41   87]]


## Experiment 3: Text and Structured Features


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score, precision_score, recall_score

categorical_columns = ["employment_type", "required_experience"]
binary_columns = ["telecommuting", "has_company_logo"]

feature_df = model_df[categorical_columns + binary_columns].copy()
feature_df["combined_text"] = combined_text
feature_df[categorical_columns] = feature_df[categorical_columns].fillna("Unknown")

X_train_structured = feature_df.loc[X_train.index]
X_val_structured = feature_df.loc[X_val.index]
X_test_structured = feature_df.loc[X_test.index]

feature_transformer = ColumnTransformer([
    ("text", TfidfVectorizer(), "combined_text"),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ("binary", SimpleImputer(strategy="most_frequent"), binary_columns),
])

structured_model = make_pipeline(
    feature_transformer,
    LinearSVC(class_weight="balanced", random_state=42)
)
structured_model.fit(X_train_structured, y_train)

structured_scores = structured_model.decision_function(X_val_structured)
threshold_results = []
for threshold in np.round(np.arange(-0.50, 0.51, 0.05), 2):
    predictions = (structured_scores > threshold).astype(int)
    _, false_alarms, missed, detected = confusion_matrix(y_val, predictions, labels=[0, 1]).ravel()
    threshold_results.append({
        "Threshold": threshold,
        "Fake precision": precision_score(y_val, predictions, pos_label=1, zero_division=0),
        "Fake recall": recall_score(y_val, predictions, pos_label=1),
        "Fake F1": f1_score(y_val, predictions, pos_label=1),
        "Detected": detected,
        "Missed": missed,
        "False alarms": false_alarms,
    })

threshold_summary = pd.DataFrame(threshold_results)
display(threshold_summary.round(3))

structured_threshold = threshold_summary.loc[threshold_summary["Fake F1"].idxmax(), "Threshold"]
best_predictions = (structured_scores > structured_threshold).astype(int)

print(f"Selected threshold: {structured_threshold:+.2f}")
print(classification_report(
    y_val, best_predictions, labels=[0, 1],
    target_names=["Real", "Fake"], digits=3, zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_val, best_predictions, labels=[0, 1]))

,Threshold,Fake precision,Fake recall,Fake F1,Detected,Missed,False alarms
0,-0.50,0.487,0.906,0.634,116,12,122
1,-0.45,0.505,0.867,0.638,111,17,109
2,-0.40,0.608,0.859,0.712,110,18,71
3,-0.35,0.655,0.859,0.743,110,18,58
4,-0.30,0.696,0.859,0.769,110,18,48
5,-0.25,0.720,0.844,0.777,108,20,42
6,-0.20,0.746,0.805,0.774,103,25,35
7,-0.15,0.764,0.758,0.761,97,31,30
8,-0.10,0.808,0.758,0.782,97,31,23
9,-0.05,0.857,0.750,0.800,96,32,16


Selected threshold: -0.05
              precision    recall  f1-score   support

        Real      0.989     0.995     0.992      3023
        Fake      0.857     0.750     0.800       128

    accuracy                          0.985      3151
   macro avg      0.923     0.872     0.896      3151
weighted avg      0.984     0.985     0.984      3151

Confusion matrix:
[[3007   16]
 [  32   96]]


In [14]:
import joblib
import sklearn
from google.colab import files

structured_model_path = "fake_job_structured_model.joblib"

joblib.dump(
    {
        "pipeline": structured_model,
        "threshold": structured_threshold,
        "sklearn_version": sklearn.__version__,
        "text_columns": text_columns,
        "categorical_columns": categorical_columns,
        "binary_columns": binary_columns,
    },
    structured_model_path,
    compress=3,
)

print("Model saved:", structured_model_path)
files.download(structured_model_path)

Model saved: fake_job_structured_model.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import joblib
import numpy as np

saved_model = joblib.load("fake_job_structured_model.joblib")
sample = X_val_structured.iloc[[0]]

original_score = structured_model.decision_function(sample)[0]
saved_score = saved_model["pipeline"].decision_function(sample)[0]

assert np.isclose(original_score, saved_score)
assert saved_model["threshold"] == structured_threshold

print(f"Original score: {original_score:.6f}")
print(f"Saved model score: {saved_score:.6f}")
print(f"Decision threshold: {saved_model['threshold']:.2f}")
print("Model export verified.")

Original score: -0.780131
Saved model score: -0.780131
Decision threshold: -0.05
Model export verified.


In [16]:
real_indices = y_train[y_train == 0].index
fake_labels = y_train[y_train == 1]

extra_fake_indices = fake_labels.sample(
    n=len(real_indices) - len(fake_labels),
    replace=True,
    random_state=42
).index

oversampled_indices = y_train.index.append(extra_fake_indices)
X_train_oversampled = X_train_structured.loc[oversampled_indices]
y_train_oversampled = y_train.loc[oversampled_indices]

from sklearn.base import clone

oversampled_model = clone(structured_model)
oversampled_model.set_params(linearsvc__class_weight=None)
oversampled_model.fit(X_train_oversampled, y_train_oversampled)

oversampled_predictions = oversampled_model.predict(X_val_structured)

print("Training class counts after oversampling:")
print(y_train_oversampled.value_counts().sort_index())

print("\nValidation report:")
print(classification_report(
    y_val,
    oversampled_predictions,
    labels=[0, 1],
    target_names=["Real", "Fake"],
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, oversampled_predictions, labels=[0, 1]))

Training class counts after oversampling:
fraudulent
0    11254
1    11254
Name: count, dtype: int64

Validation report:
              precision    recall  f1-score   support

        Real      0.987     0.995     0.991      3023
        Fake      0.863     0.688     0.765       128

    accuracy                          0.983      3151
   macro avg      0.925     0.841     0.878      3151
weighted avg      0.982     0.983     0.982      3151

Confusion matrix:
[[3009   14]
 [  40   88]]


In [17]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

split_labels = pd.concat([
    pd.Series("train", index=X_train.index),
    pd.Series("val", index=X_val.index),
    pd.Series("test", index=X_test.index),
])

check_df = model_df[["title", "description", "fraudulent"]].copy()
check_df["split"] = split_labels
check_df["normalized_description"] = (
    check_df["description"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)

group_stats = check_df.groupby("normalized_description").agg(
    rows=("split", "size"),
    splits=("split", "nunique"),
    labels=("fraudulent", "nunique"),
)
repeated_groups = group_stats[group_stats["rows"] > 1]
cross_split_groups = repeated_groups[repeated_groups["splits"] > 1]

print("Groups with repeated description:", len(repeated_groups))
print("Rows in repeated groups:", int(repeated_groups["rows"].sum()))
print("Groups spanning multiple splits:", len(cross_split_groups))

cross_split_rows = check_df[check_df["normalized_description"].isin(cross_split_groups.index)]
print("\nRows in cross-split groups by split and label:")
print(cross_split_rows.groupby(["split", "fraudulent"]).size().unstack(fill_value=0))

train_texts = set(check_df.loc[check_df["split"] == "train", "normalized_description"])
for split_name in ["val", "test"]:
    part = check_df[check_df["split"] == split_name]
    overlap = part["normalized_description"].isin(train_texts)
    print(
        f"{split_name}: {overlap.sum()}/{len(part)} rows share exact description with train, "
        f"fake among them: {int(part.loc[overlap, 'fraudulent'].sum())}"
    )

conflict_texts = group_stats[group_stats["labels"] > 1].index
conflicting = check_df[check_df["normalized_description"].isin(conflict_texts)].copy()
conflicting["description_preview"] = conflicting["description"].str[:80]
print("\nGroups with conflicting labels:", len(conflict_texts))
print("Rows in conflicting groups:", len(conflicting))
display(conflicting.sort_values("normalized_description")[
    ["split", "fraudulent", "title", "description_preview"]
].head(20))

near_vectorizer = TfidfVectorizer()
train_matrix = near_vectorizer.fit_transform(
    check_df.loc[check_df["split"] == "train", "normalized_description"]
)
neighbors = NearestNeighbors(n_neighbors=1, metric="cosine").fit(train_matrix)

near_results = []
for split_name in ["val", "test"]:
    part = check_df[check_df["split"] == split_name]
    distances, _ = neighbors.kneighbors(near_vectorizer.transform(part["normalized_description"]))
    similarity = 1 - distances[:, 0]
    for cutoff in [0.80, 0.90, 0.95, 0.99]:
        mask = similarity >= cutoff
        near_results.append({
            "Split": split_name,
            "Similarity >=": cutoff,
            "Rows": int(mask.sum()),
            "Share %": round(mask.mean() * 100, 2),
            "Fake rows": int(part["fraudulent"].values[mask].sum()),
        })

display(pd.DataFrame(near_results))

Groups with repeated description: 1127
Rows in repeated groups: 4133
Groups spanning multiple splits: 0

Rows in cross-split groups by split and label:
Empty DataFrame
Columns: []
Index: []
val: 0/3151 rows share exact description with train, fake among them: 0
test: 0/2597 rows share exact description with train, fake among them: 0

Groups with conflicting labels: 1
Rows in conflicting groups: 3


,split,fraudulent,title,description_preview
9138,train,1,Front-End/JavaScript Engineer,Workable is a product-driven software company....
11889,train,0,Developer (Front-end team),Workable is a product-driven software company....
15236,train,0,Developer,Workable is a product-driven software company....


,Split,Similarity >=,Rows,Share %,Fake rows
0,val,0.80,880,27.93,15
1,val,0.90,58,1.84,2
2,val,0.95,0,0.00,0
3,val,0.99,0,0.00,0
4,test,0.80,298,11.47,4
5,test,0.90,32,1.23,2
6,test,0.95,2,0.08,0
7,test,0.99,0,0.00,0


In [18]:
extended_text_columns = ["title", "company_profile", "description", "requirements", "benefits"]
extended_categorical_columns = [
    "employment_type", "required_experience", "required_education", "function", "industry"
]
extended_binary_columns = ["telecommuting", "has_company_logo", "has_questions", "has_salary_range"]

extended_df = model_df[extended_categorical_columns].fillna("Unknown")
extended_df["combined_text"] = model_df[extended_text_columns].fillna("").agg(" ".join, axis=1)
extended_df["telecommuting"] = model_df["telecommuting"]
extended_df["has_company_logo"] = model_df["has_company_logo"]
extended_df["has_questions"] = model_df["has_questions"]
extended_df["has_salary_range"] = model_df["salary_range"].notna().astype(int)

X_train_extended = extended_df.loc[X_train.index]
X_val_extended = extended_df.loc[X_val.index]
X_test_extended = extended_df.loc[X_test.index]

extended_transformer = ColumnTransformer([
    ("text", TfidfVectorizer(), "combined_text"),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), extended_categorical_columns),
    ("binary", "passthrough", extended_binary_columns),
])

extended_model = make_pipeline(
    extended_transformer,
    LinearSVC(class_weight="balanced", random_state=42)
)
extended_model.fit(X_train_extended, y_train)

extended_scores = extended_model.decision_function(X_val_extended)
extended_results = []
for threshold in np.round(np.arange(-0.50, 0.51, 0.05), 2):
    predictions = (extended_scores > threshold).astype(int)
    _, false_alarms, missed, detected = confusion_matrix(y_val, predictions, labels=[0, 1]).ravel()
    extended_results.append({
        "Threshold": threshold,
        "Fake precision": precision_score(y_val, predictions, pos_label=1, zero_division=0),
        "Fake recall": recall_score(y_val, predictions, pos_label=1),
        "Fake F1": f1_score(y_val, predictions, pos_label=1),
        "Detected": detected,
        "Missed": missed,
        "False alarms": false_alarms,
    })

extended_summary = pd.DataFrame(extended_results)
display(extended_summary.round(3))

extended_threshold = extended_summary.loc[extended_summary["Fake F1"].idxmax(), "Threshold"]
extended_predictions = (extended_scores > extended_threshold).astype(int)

print(f"Selected threshold: {extended_threshold:+.2f}")
print(classification_report(
    y_val, extended_predictions, labels=[0, 1],
    target_names=["Real", "Fake"], digits=3, zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_val, extended_predictions, labels=[0, 1]))

reference_f1 = f1_score(y_val, (structured_scores > structured_threshold).astype(int), pos_label=1)
extended_f1 = f1_score(y_val, extended_predictions, pos_label=1)
print(f"Reference validation Fake F1: {reference_f1:.3f}")
print(f"Extended validation Fake F1: {extended_f1:.3f}")
print(f"Difference: {extended_f1 - reference_f1:+.3f}")

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,Threshold,Fake precision,Fake recall,Fake F1,Detected,Missed,False alarms
0,-0.50,0.498,0.891,0.639,114,14,115
1,-0.45,0.531,0.883,0.663,113,15,100
2,-0.40,0.539,0.859,0.663,110,18,94
3,-0.35,0.556,0.852,0.673,109,19,87
4,-0.30,0.583,0.852,0.692,109,19,78
5,-0.25,0.623,0.812,0.705,104,24,63
6,-0.20,0.642,0.797,0.711,102,26,57
7,-0.15,0.669,0.789,0.724,101,27,50
8,-0.10,0.756,0.703,0.729,90,38,29
9,-0.05,0.869,0.672,0.758,86,42,13


Selected threshold: -0.05
              precision    recall  f1-score   support

        Real      0.986     0.996     0.991      3023
        Fake      0.869     0.672     0.758       128

    accuracy                          0.983      3151
   macro avg      0.927     0.834     0.874      3151
weighted avg      0.981     0.983     0.981      3151

Confusion matrix:
[[3010   13]
 [  42   86]]
Reference validation Fake F1: 0.800
Extended validation Fake F1: 0.758
Difference: -0.042


In [19]:
from sklearn.metrics import roc_auc_score

final_test_scores = structured_model.decision_function(X_test_structured)
final_test_predictions = (final_test_scores > structured_threshold).astype(int)

print(f"Final threshold: {structured_threshold:+.2f}")
print(classification_report(
    y_test, final_test_predictions, labels=[0, 1],
    target_names=["Real", "Fake"], digits=3, zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_test, final_test_predictions, labels=[0, 1]))
print(f"ROC AUC: {roc_auc_score(y_test, final_test_scores):.3f}")

Final threshold: -0.05
              precision    recall  f1-score   support

        Real      0.987     0.994     0.990      2466
        Fake      0.860     0.748     0.800       131

    accuracy                          0.981      2597
   macro avg      0.923     0.871     0.895      2597
weighted avg      0.980     0.981     0.981      2597

Confusion matrix:
[[2450   16]
 [  33   98]]
ROC AUC: 0.973


In [20]:
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve

comparison_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=42),
}

def model_scores(model, features):
    if hasattr(model, "decision_function"):
        return model.decision_function(features)
    log_probabilities = model.predict_log_proba(features)
    return log_probabilities[:, 1] - log_probabilities[:, 0]

comparison_rows = []
for name, classifier in comparison_models.items():
    pipeline = make_pipeline(clone(feature_transformer), classifier)
    pipeline.fit(X_train_structured, y_train)

    val_scores = model_scores(pipeline, X_val_structured)
    precisions, recalls, thresholds = precision_recall_curve(y_val, val_scores)
    f1_values = 2 * precisions[:-1] * recalls[:-1] / np.clip(precisions[:-1] + recalls[:-1], 1e-12, None)
    best_threshold = thresholds[np.argmax(f1_values)]

    test_scores = model_scores(pipeline, X_test_structured)
    for split_name, labels, scores in [("Validation", y_val, val_scores), ("Test", y_test, test_scores)]:
        predictions = (scores >= best_threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
        comparison_rows.append({
            "Model": name,
            "Split": split_name,
            "Accuracy": (tp + tn) / len(labels),
            "Fake precision": precision_score(labels, predictions, pos_label=1, zero_division=0),
            "Fake recall": recall_score(labels, predictions, pos_label=1),
            "Fake F1": f1_score(labels, predictions, pos_label=1),
            "ROC AUC": roc_auc_score(labels, scores),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        })

comparison_summary = pd.DataFrame(comparison_rows)
display(comparison_summary.round(3))

,Model,Split,Accuracy,Fake precision,Fake recall,Fake F1,ROC AUC,TN,FP,FN,TP
0,Logistic Regression,Validation,0.981,0.793,0.719,0.754,0.985,2999,24,36,92
1,Logistic Regression,Test,0.979,0.824,0.748,0.784,0.980,2445,21,33,98
2,Multinomial Naive Bayes,Validation,0.907,0.266,0.734,0.391,0.889,2764,259,34,94
3,Multinomial Naive Bayes,Test,0.883,0.249,0.656,0.361,0.894,2206,260,45,86
4,Linear SVM,Validation,0.985,0.872,0.742,0.802,0.978,3009,14,33,95
5,Linear SVM,Test,0.982,0.874,0.740,0.802,0.973,2452,14,34,97
